# **NOTEBOOK2: Implementing SHAP into different blackbox models, syntehtic data**

STEP 1: generate data and clean it

In [ ]:
import pandas as pd
import numpy as np


def generate_data(number_samples, number_features, noise):
    d={}
    l=[]
    f1=np.random.randn(number_samples)
    d["f1"]=f1
    for i in range(2,number_features+1):
        alpha_i= np.random.randn()
        d["f%d"%i]=alpha_i*d["f%d"%(i-1)]+noise
        l.append(alpha_i)
    df=pd.DataFrame(d)
    return (df,l)



generate_data(10,3,0.2)



In [ ]:
data_tot=generate_data(150,5,0.2)
data=data_tot[0]

data.head()
data.columns


STEP 2: training a model on the data:



In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split


X=data.drop(["f5"], axis=1)
y=data.f5
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)
model=RandomForestRegressor(random_state=55)
model.fit(train_X,train_y)
preds=model.predict(val_X)

mae=mean_absolute_error(preds,val_y)

print(mae)

apply SHAP


In [ ]:
import shap

print(data_tot[1])

shap.initjs()

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(val_X)


shap.summary_plot(shap_values, val_X)


shap.summary_plot(shap_values, val_X, plot_type="bar")


shap.force_plot(explainer.expected_value, shap_values[0], val_X.iloc[0])



Test with some decorrelated data




In [ ]:
d={}
l=[]
f1=np.random.randn(150)
d["f1"]=f1
for i in range(2,5+1):
    alpha_i= np.random.randn()
    d["f%d"%i]=alpha_i*d["f%d"%(i-1)]+0.2
    l.append(alpha_i)
d["f6"]=np.random.randn(150)
df=pd.DataFrame(d)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split


X=df.drop(["f5"], axis=1)
y=df.f5
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)
model=RandomForestRegressor(random_state=55)
model.fit(train_X,train_y)
preds=model.predict(val_X)

mae=mean_absolute_error(preds,val_y)

print(mae)

In [ ]:



shap.initjs()

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(val_X)


shap.summary_plot(shap_values, val_X)


shap.summary_plot(shap_values, val_X, plot_type="bar")


shap.force_plot(explainer.expected_value, shap_values[0], val_X.iloc[0])


#SHAP reconnait que f6 n'a pas eu d'importance.
maintenant, essayons de prédire f6

In [ ]:
X=df.drop(["f6"], axis=1)
y=df.f6
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)
model=RandomForestRegressor(random_state=55)
model.fit(train_X,train_y)
preds=model.predict(val_X)

mae=mean_absolute_error(preds,val_y)

print(mae)

shap.initjs()

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(val_X)


shap.summary_plot(shap_values, val_X)


shap.summary_plot(shap_values, val_X, plot_type="bar")


shap.force_plot(explainer.expected_value, shap_values[0], val_X.iloc[0])

shap trouve prédictions là où il n'y en a pas. testons avec plus de lignes

In [ ]:
d={}
l=[]
f1=np.random.randn(800)
d["f1"]=f1
for i in range(2,5+1):
    alpha_i= np.random.randn()
    d["f%d"%i]=alpha_i*d["f%d"%(i-1)]+0.2
    l.append(alpha_i)
d["f6"]=np.random.randn(800)+0.2
df=pd.DataFrame(d)


X=df.drop(["f6"], axis=1)
y=df.f6
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)
model=RandomForestRegressor(random_state=55)
model.fit(train_X,train_y)
preds=model.predict(val_X)

mae=mean_absolute_error(preds,val_y)

print(mae)

shap.initjs()

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(val_X)


shap.summary_plot(shap_values, val_X)


shap.summary_plot(shap_values, val_X, plot_type="bar")


shap.force_plot(explainer.expected_value, shap_values[0], val_X.iloc[0])

cela ne change rien. tentons les graphes de permutation. Cas d'abord simple (prédiction de f5)

In [ ]:
import eli5
from eli5.sklearn import PermutationImportance

X=df.drop(["f5"], axis=1)
y=df.f5
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)
model=RandomForestRegressor(random_state=55)
model.fit(train_X,train_y)
preds=model.predict(val_X)

mae=mean_absolute_error(preds,val_y)

print(mae)

perm = PermutationImportance(model, random_state=1).fit(val_X, val_y)
eli5.show_weights(perm, feature_names = val_X.columns.tolist())

les permutations identifient parfaitement l'inutilité de f6

In [ ]:
X=df.drop(["f6"], axis=1)
y=df.f6
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)
model=RandomForestRegressor(random_state=55)
model.fit(train_X,train_y)
preds=model.predict(val_X)

mae=mean_absolute_error(preds,val_y)


perm = PermutationImportance(model, random_state=1).fit(val_X, val_y)
eli5.show_weights(perm, feature_names = val_X.columns.tolist())


c'est mieux!!! les permutations arrivent bien à détecter quelle feature est inutile. Tentons LIME:

In [ ]:
import lime
import lime.lime_tabular

X=df.drop(["f5"], axis=1)
y=df.f5
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)
model=RandomForestRegressor(random_state=55)
model.fit(train_X,train_y)
preds=model.predict(val_X)

explainer = lime.lime_tabular.LimeTabularExplainer(train_X.values, feature_names=train_X.columns.values.tolist(),  verbose=True, mode='regression')

exp = explainer.explain_instance(val_X.values[5], model.predict, num_features=6)

html = exp.as_html(show_table=True)
with open("lime_explanation.html", "w") as f:
    f.write(html)




LIME seems off. the results vary widly between rows of data. Maybe it comes from the non-linearity of the random forest? let's try XGRboost


In [ ]:

from sklearn.linear_model import LinearRegression

import lime
import lime.lime_tabular

X=df.drop(["f5"], axis=1)
y=df.f5
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)
model=LinearRegression()
model.fit(train_X,train_y)
preds=model.predict(val_X)

explainer = lime.lime_tabular.LimeTabularExplainer(train_X.values, feature_names=train_X.columns.values.tolist(),  verbose=True, mode='regression')

exp = explainer.explain_instance(val_X.values[0], model.predict, num_features=6)

html = exp.as_html(show_table=True)
with open("lime_explanation.html", "w") as f:
    f.write(html)



after trying lightgbm, random forest and linear regression models, LIME appears to be inefficient identifying f6 as the useless columns. linear regression can offer exact coefficients. what could LIME be used for ? 

## SUMMARY #


General observations:
- permutation importance matrices seem to be the most efficient method when it comes to identify which features are useless and when a target feature cannot be predicted.
- SHAP gives satisfactory results identify a useless variable, but nothing when a target feature cannot be predicted
- LIME does not provide answers in both cases

Next step: play with the data and and other configurations (simpler linear models, or try linear combinations of lines)


#### STEP 1

In [ ]:
d={}
f1=np.random.randn(800)
d["f1"]=f1
for i in range(2,5+1):
    d["f%d"%i]=np.random.randn(800)
d["f6"]=0.5*d["f1"]+2*d["f2"]+1000*d["f3"]-0.1*d["f4"]
df=pd.DataFrame(d)


X=df.drop(["f6"], axis=1)
y=df.f6
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)
model=RandomForestRegressor(random_state=55)
model.fit(train_X,train_y)
preds=model.predict(val_X)

mae=mean_absolute_error(preds,val_y)

print(mae)

shap.initjs()

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(val_X)


shap.summary_plot(shap_values, val_X)


shap.summary_plot(shap_values, val_X, plot_type="bar")


shap.force_plot(explainer.expected_value, shap_values[0], val_X.iloc[0])

works identifying f3 when the coefficient is huge. let's try a more balanced linear combination

In [ ]:
d={}
f1=np.random.randn(800)
d["f1"]=f1
for i in range(2,5+1):
    d["f%d"%i]=np.random.randn(800)
d["f6"]=0.6*d["f1"]+0.1*d["f2"]+0.2*d["f3"]-0.1*d["f4"]+3
df=pd.DataFrame(d)


X=df.drop(["f6"], axis=1)
y=df.f6
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)
model=RandomForestRegressor(random_state=55)
model.fit(train_X,train_y)
preds=model.predict(val_X)

mae=mean_absolute_error(preds,val_y)

print(mae)

shap.initjs()

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(val_X)


shap.summary_plot(shap_values, val_X)


shap.summary_plot(shap_values, val_X, plot_type="bar")


shap.force_plot(explainer.expected_value, shap_values[0], val_X.iloc[0])

SHAP semble très compétent pour identifier quelles features sont importantes dans le cas d'une combinaison linéaire de features. Avec LIME


In [ ]:
d={}
f1=np.random.randn(800)
d["f1"]=f1
for i in range(2,5+1):
    d["f%d"%i]=np.random.randn(800)
d["f6"]=0.5*d["f1"]+2*d["f2"]+1000*d["f3"]-0.1*d["f4"]
df=pd.DataFrame(d)


X=df.drop(["f6"], axis=1)
y=df.f6
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)
model=RandomForestRegressor(random_state=55)
model.fit(train_X,train_y)
preds=model.predict(val_X)

explainer = lime.lime_tabular.LimeTabularExplainer(train_X.values, feature_names=train_X.columns.values.tolist(),  verbose=True, mode='regression')

exp = explainer.explain_instance(val_X.values[0], model.predict, num_features=6)

html = exp.as_html(show_table=True)
with open("lime_explanation.html", "w") as f:
    f.write(html)

less efficient. let's try a different approach: make a confounding variable and see if SHAP notices. 

In [ ]:
d={}
f1=np.random.randn(800)
d["f1"]=f1
d["f2"]=d["f1"]*10+2
for i in range(3,5+1):
    d["f%d"%i]=np.random.randn(800)
d["f6"]=0.6*d["f1"]+0.1*d["f2"]+0.2*d["f3"]-0.1*d["f4"]+3
df=pd.DataFrame(d)


X=df.drop(["f6"], axis=1)
y=df.f6
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)
model=RandomForestRegressor(random_state=55)
model.fit(train_X,train_y)
preds=model.predict(val_X)



shap.initjs()

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(val_X)


shap.summary_plot(shap_values, val_X)


shap.summary_plot(shap_values, val_X, plot_type="bar")


shap.force_plot(explainer.expected_value, shap_values[0], val_X.iloc[0])

no, SHAP does not identify the fact that f2 depends entirely on f1